In [1]:
import sys
sys.path.append("..")

from src.ingestion.gee_download import init_gee, load_config, build_aoi, get_s1_collection
from src.ingestion.gee_download import preview_composite, build_monthly_composites
from src.ingestion.grid import create_grid, tag_zones, preview_grid, save_grid
from src.ingestion.acled_fetch import load_config, fetch_acled, filter_to_aoi, incident_summary

init_gee()
config = load_config("../configs/config.yaml")


2026-05-24 21:13:57.575 | SUCCESS  | src.ingestion.gee_download:init_gee:33 - GEE initialised — project: nigeria-forest-monitor


In [2]:
aoi = build_aoi(config["aoi"]["zones"]["old_oyo_core"]["bbox"])

collection = get_s1_collection(
    aoi,
    "2024-01-01",
    "2024-03-01",
    config
)
print(collection.size().getInfo(), "images found")

2026-05-24 21:13:57.737 | INFO     | src.ingestion.gee_download:build_aoi:60 - AOI: lon [3.6, 4.6] lat [8.2, 9.2]
2026-05-24 21:13:58.690 | INFO     | src.ingestion.gee_download:get_s1_collection:96 - Found 14 Sentinel-1 images | 2024-01-01 → 2024-03-01


14 images found


In [3]:
composites = build_monthly_composites(collection, "2024-01-01", "2024-03-01", aoi)

# Preview January composite
m = preview_composite(composites[0]["image"], aoi, label="Old Oyo — Jan 2024 VV")
m

2026-05-24 21:14:00.216 | INFO     | src.ingestion.gee_download:build_monthly_composites:142 - Composite 2024-01 built from 8 images
2026-05-24 21:14:00.904 | INFO     | src.ingestion.gee_download:build_monthly_composites:142 - Composite 2024-02 built from 6 images
2026-05-24 21:14:00.917 | SUCCESS  | src.ingestion.gee_download:build_monthly_composites:145 - Built 2 monthly composites


Map(center=[8.700103597702125, 4.100000000000248], controls=(WidgetControl(options=['position', 'transparent_b…

In [4]:
# Build + tag grid
grid = create_grid(config, zone="full")
grid = tag_zones(grid, config)

# Summary
print(grid.groupby("zone")["cell_id"].count().rename("cells"))
print(f"\nTotal cells: {len(grid)}")

# Visualise
m = preview_grid(grid, config, color_by_zone=True)
m

2026-05-24 21:14:03.803 | SUCCESS  | src.ingestion.grid:create_grid:76 - Grid created: 2352 cells | res=0.05° (~5.6 km) | zone=full
2026-05-24 21:14:04.587 | INFO     | src.ingestion.grid:tag_zones:103 - Zone tagging: {'outside': 1480, 'old_oyo_core': 336, 'kwara_border': 312, 'kainji_link': 224}


zone
kainji_link      224
kwara_border     312
old_oyo_core     336
outside         1480
Name: cells, dtype: int64

Total cells: 2352


2026-05-24 21:14:05.714 | INFO     | src.ingestion.grid:preview_grid:259 - Grid map rendered: 2352 cells


Map(center=[9.0, 4.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [5]:
df        = fetch_acled(config, start_year=2019, end_year=2025)

# Normalise location spellings
df["location"] = df["location"].str.strip().str.title()
df["location"] = df["location"].replace({
    "Illorin": "Ilorin",
    "Ilorin ": "Ilorin"
})

incidents = filter_to_aoi(df, config)
incident_summary(incidents)

2026-05-24 21:14:05.847 | INFO     | src.ingestion.acled_fetch:get_acled_token:49 - Requesting ACLED OAuth token...
2026-05-24 21:14:17.406 | SUCCESS  | src.ingestion.acled_fetch:get_acled_token:65 - ACLED token obtained (valid 24h)
2026-05-24 21:14:17.413 | INFO     | src.ingestion.acled_fetch:fetch_acled:116 - Fetching ACLED data for Nigeria (2019-2025)...
2026-05-24 21:14:29.567 | SUCCESS  | src.ingestion.acled_fetch:fetch_acled:139 - Fetched 3,451 ACLED incidents
2026-05-24 21:14:29.584 | SUCCESS  | src.ingestion.acled_fetch:filter_to_aoi:170 - Filtered to AOI: 41 incidents (from 3,451 total Nigeria records)



ACLED INCIDENT SUMMARY - Nigeria Forest Corridor
Total incidents : 41
Date range      : 2019-01-13 -> 2025-05-13
Total fatalities: 53

By event type:
event_type
Violence against civilians    24
Battles                       17

Top 10 locations:
location
Ilorin         5
Kemanji        2
Babanla        2
Ifon           2
Ilobu          2
Odo-Eku        2
Isanlu-Isin    2
Igogo          1
Karunji        1
Ire            1


In [6]:
df.head()

,event_date,event_type,sub_event_type,actor1,location,latitude,longitude,notes,fatalities
0,2019-04-14,Violence against civilians,Attack,Unidentified Communal Militia (Nigeria),Akwanga,8.9167,8.3833,14 April: Gunmen attacked a group of people wh...,16
1,2019-04-15,Violence against civilians,Abduction/forced disappearance,Unidentified Armed Group (Nigeria),Iwara,7.5333,4.7167,15 April: An Ibadan based lawyer was kidnapped...,0
2,2019-02-04,Violence against civilians,Abduction/forced disappearance,Unidentified Armed Group (Nigeria),Birnin Magaji,12.5592,6.8946,04 February. Unidentified Armed Group abducted...,0
3,2019-04-13,Violence against civilians,Attack,Unidentified Armed Group (Nigeria),Yandev,7.3667,9.0500,13 April: Unidentified armed group killed the ...,1
4,2019-04-16,Violence against civilians,Attack,Unidentified Communal Militia (Nigeria),Kwande,6.8010,9.4702,16 April: Unidentified armed pastoralists kill...,2


In [7]:
from src.ingestion.gee_download import init_gee, load_config, build_aoi, get_s1_collection
from src.preprocessing.speckle_filter import refined_lee_filter, preview_filter_comparison


In [8]:
init_gee()

aoi    = build_aoi(config["aoi"]["zones"]["old_oyo_core"]["bbox"])
s1     = get_s1_collection(aoi, "2024-01-01", "2024-03-01", config)

# Filter a single image
raw      = s1.first()
filtered = refined_lee_filter(raw)

# Visual comparison — toggle layers on the map
m = preview_filter_comparison(raw, aoi)
m

2026-05-24 21:14:37.140 | SUCCESS  | src.ingestion.gee_download:init_gee:33 - GEE initialised — project: nigeria-forest-monitor
2026-05-24 21:14:37.140 | INFO     | src.ingestion.gee_download:build_aoi:60 - AOI: lon [3.6, 4.6] lat [8.2, 9.2]
2026-05-24 21:14:44.257 | INFO     | src.ingestion.gee_download:get_s1_collection:96 - Found 14 Sentinel-1 images | 2024-01-01 → 2024-03-01
2026-05-24 21:14:48.756 | INFO     | src.preprocessing.speckle_filter:preview_filter_comparison:218 - Comparison map ready — toggle layers to compare


Map(center=[8.700103597702125, 4.100000000000248], controls=(WidgetControl(options=['position', 'transparent_b…

In [9]:
from src.ingestion.gee_download import init_gee, load_config, build_aoi
from src.preprocessing.baseline import build_baseline, compute_baseline_stats, preview_baseline

# init_gee()
# config   = load_config("configs/config.yaml")
aoi      = build_aoi(config["aoi"]["zones"]["old_oyo_core"]["bbox"])

# Build baseline (this may take 30-60 seconds — GEE is processing server-side)
baseline = build_baseline(aoi, config, filter_method="lee")

# Preview
m = preview_baseline(baseline, aoi)
m

2026-05-24 21:14:50.643 | INFO     | src.ingestion.gee_download:build_aoi:60 - AOI: lon [3.6, 4.6] lat [8.2, 9.2]
2026-05-24 21:14:50.643 | INFO     | src.preprocessing.baseline:build_baseline:51 - Building baseline: 2020-01-01 → 2022-12-31
2026-05-24 21:14:51.705 | INFO     | src.ingestion.gee_download:get_s1_collection:96 - Found 362 Sentinel-1 images | 2020-01-01 → 2022-12-31
2026-05-24 21:14:52.791 | INFO     | src.preprocessing.baseline:build_baseline:70 - Baseline collection: 362 images
2026-05-24 21:14:52.793 | INFO     | src.preprocessing.speckle_filter:apply_filter_to_collection:170 - Applying lee filter to collection (server-side)...
2026-05-24 21:14:52.798 | SUCCESS  | src.preprocessing.speckle_filter:apply_filter_to_collection:172 - Speckle filter applied
2026-05-24 21:14:52.800 | SUCCESS  | src.preprocessing.baseline:build_baseline:78 - Baseline mosaic built from 362 images | filter: lee
2026-05-24 21:14:57.703 | INFO     | src.preprocessing.baseline:preview_baseline:232 -

Map(center=[8.700103597702125, 4.100000000000248], controls=(WidgetControl(options=['position', 'transparent_b…